# Control Basic Models

**Author:** Lias
**Task:** Compute RDMs from three simple, non-neural baselines on the 1000 shared NSD images:
1. **Pixel space** — raw pixel values flattened into a vector
2. **PCA** — pixel vectors reduced to 50 principal components
3. **Linear regression** — predict fMRI activity from pixel features (encoding model)

These are *control* models: no deep learning involved. If the beta-VAE outperforms them,
that shows learned representations are doing something the raw pixels cannot.

The key output of each baseline is an **RDM** (Representational Dissimilarity Matrix):
a 1000×1000 matrix where entry [i,j] = how different the model thinks images i and j are.
We will compare these model RDMs to the brain RDMs using RSA (Representational Similarity Analysis).

## 0. Setup — imports and paths

In [ ]:
# === COLAB SETUP (skip if running locally) ===
# !pip install git+https://github.com/lucas-nunn/PSM-NeuroAI-Final.git@YOUR_BRANCH_NAME
# from google.colab import drive
# drive.mount('/content/drive/', force_remount=True)
# DATA_DIR = '/content/drive/MyDrive/YOUR_DATA_FOLDER'

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

# Import shared plotting helper from the project package
from psm_final.helpers.plotting import plot_rsa

# ── PATHS ────────────────────────────────────────────────────────────────────
# Adjust DATA_DIR to wherever the NSD / Algonauts data lives on your machine
DATA_DIR    = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

STIM_DIR = os.path.join(DATA_DIR, "stimuli")    # 1000 shared .png images
FMRI_DIR = os.path.join(DATA_DIR, "fmri_data")  # per-subject fMRI files

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete.")

## 1. Load the 1000 shared images

The NSD / Algonauts dataset has 1000 images that **every subject viewed** — these are our test images.
All model-vs-brain comparisons are done on this shared set.

In [ ]:
img_paths = sorted(glob(os.path.join(STIM_DIR, "*.png")))
N_IMAGES  = len(img_paths)
print(f"Found {N_IMAGES} images in {STIM_DIR}")

# Resize all images to the same size so we can stack them into a matrix.
# 64×64 is a good balance between visual detail and computational cost.
IMG_SIZE = (64, 64)

def load_image(path, size=IMG_SIZE):
    """Load one image, resize it, return float32 array (H, W, 3) in [0, 1]."""
    img = Image.open(path).convert("RGB")
    img = img.resize(size, Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0

print(f"Loading {N_IMAGES} images at {IMG_SIZE}...")
images = np.stack([load_image(p) for p in img_paths])   # shape: (N, H, W, 3)
print(f"Images array shape: {images.shape}")

# Plot a few examples to check everything loaded correctly
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(f"img {i}")
    ax.axis("off")
plt.suptitle("Sample images from the 1000-image shared set")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "control_basic_sample_images.png"), dpi=100)
plt.show()

## 2. Baseline 1: Pixel space RDM

### What is it?
The simplest possible model: raw pixel values.
Each image becomes a flat vector of length H × W × 3 = 64×64×3 = 12 288 numbers.

### Distance metric: correlation distance
`distance(i, j) = 1 − Pearson_correlation(pixels_i, pixels_j)`

Two identical images → distance 0. Completely uncorrelated images → distance ≈ 1.
This is the standard metric used in RSA in neuroscience.

### Why bother?
This is the lower bound. If the brain RDM matches *only* the pixel RDM, the brain just responds
to low-level statistics (brightness, colour). If the beta-VAE does better, something more abstract
is being captured.

In [ ]:
# Flatten images: (N, H, W, 3) → (N, H*W*3)
pixels = images.reshape(N_IMAGES, -1)
print(f"Pixel feature matrix shape: {pixels.shape}")

# Compute pairwise correlation distances between all rows → upper triangle vector
pixel_rdm_vec = pdist(pixels, metric="correlation")
pixel_rdm     = squareform(pixel_rdm_vec)   # full N×N symmetric matrix
print(f"Pixel RDM shape: {pixel_rdm.shape}")

# Save both the full matrix and the upper-triangle vector
np.save(os.path.join(RESULTS_DIR, "rdm_pixel.npy"),     pixel_rdm)
np.save(os.path.join(RESULTS_DIR, "rdm_pixel_vec.npy"), pixel_rdm_vec)
print("Saved pixel RDM to results/")

# Visualise
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.matshow(pixel_rdm, cmap="viridis")
plt.colorbar(im, ax=ax)
ax.set_title("Pixel space RDM (correlation distance)")
ax.set_xlabel("Image index")
ax.set_ylabel("Image index")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rdm_pixel.png"), dpi=100)
plt.show()

## 3. Baseline 2: PCA RDM

### What is it?
We apply **PCA** to the pixel vectors, keeping the top 50 principal components that explain
most of the variance. Then we compute the RDM on the reduced vectors.

### Why?
- Pixel space has 12 288 dimensions, many of which are redundant or noisy.
- PCA keeps the most important directions of variation.
- 50 components is the standard number used in RSA pipelines (see `fit_pca` in the repo).

### Why standardise before PCA?
PCA is sensitive to feature scales — if one pixel channel has bigger values it would dominate.
`StandardScaler` makes each pixel dimension have mean=0 and std=1 before PCA is applied.

In [ ]:
# Standardise pixel features (mean=0, std=1 per feature)
scaler = StandardScaler()
pixels_scaled = scaler.fit_transform(pixels)

N_COMPONENTS = 50
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pixels_pca = pca.fit_transform(pixels_scaled)  # shape: (N, 50)
print(f"PCA output shape: {pixels_pca.shape}")

# How much variance do we explain?
explained = np.cumsum(pca.explained_variance_ratio_)
print(f"Variance explained by {N_COMPONENTS} PCs: {explained[-1]*100:.1f}%")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, N_COMPONENTS + 1), explained * 100, marker="o", ms=4)
plt.axhline(90, color="red", linestyle="--", label="90% threshold")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance (%)")
plt.title("PCA: Cumulative explained variance on pixel features")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "pca_variance_explained.png"), dpi=100)
plt.show()

In [ ]:
# Compute PCA RDM
pca_rdm_vec = pdist(pixels_pca, metric="correlation")
pca_rdm     = squareform(pca_rdm_vec)
print(f"PCA RDM shape: {pca_rdm.shape}")

np.save(os.path.join(RESULTS_DIR, "rdm_pca.npy"),     pca_rdm)
np.save(os.path.join(RESULTS_DIR, "rdm_pca_vec.npy"), pca_rdm_vec)
print("Saved PCA RDM to results/")

# Side-by-side comparison: pixel vs. PCA
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].matshow(pixel_rdm, cmap="viridis")
plt.colorbar(im0, ax=axes[0])
axes[0].set_title("Pixel space RDM")
im1 = axes[1].matshow(pca_rdm, cmap="viridis")
plt.colorbar(im1, ax=axes[1])
axes[1].set_title(f"PCA RDM (50 components)")
for ax in axes:
    ax.set_xlabel("Image index")
    ax.set_ylabel("Image index")
plt.suptitle("Pixel vs. PCA RDMs")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rdm_pixel_vs_pca.png"), dpi=100)
plt.show()

# Sanity check
r, p = spearmanr(pixel_rdm_vec, pca_rdm_vec)
print(f"Spearman RSA: pixel vs. PCA RDM = r={r:.3f}, p={p:.2e}")
print("(Expected: high — PCA is derived from pixels)")

## 4. Baseline 3: Linear regression (encoding model)

### What is it?
An **encoding model** tries to *predict* the brain's fMRI response to each image from the model's features.
Here we use **Ridge regression** (linear regression + regularisation) to predict fMRI from the 50 PCA features.

### Why Ridge and not plain linear regression?
Ridge adds a penalty `alpha × ||weights||²` for large weights, which prevents overfitting when you have
many features (50 PCs) and relatively few observations. It is standard in neuroscience encoding models.

### What does this give us?
- A *predicted* brain response for each image, based only on pixel information
- An RDM computed from these predicted responses
- If the beta-VAE correlates more with the brain than this linear baseline, that is the key finding

> **Note:** This section requires fMRI data. If you don't have it yet, the cell below will detect
> this and skip gracefully — you can come back to it later.

In [ ]:
def get_roi_mask(roi, hemisphere, subj_dir):
    """Load a binary ROI mask. Adapted from the week 7 course notebook."""
    roi_class_map = {
        ("V1v","V1d","V2v","V2d","V3v","V3d","hV4"): "prf-visualrois",
        ("EBA","FBA-1","FBA-2","mTL-bodies"): "floc-bodies",
        ("OFA","FFA-1","FFA-2","mTL-faces","aTL-faces"): "floc-faces",
        ("OPA","PPA","RSC"): "floc-places",
        ("OWFA","VWFA-1","VWFA-2","mfs-words","mTL-words"): "floc-words",
        ("early","midventral","midlateral","midparietal","ventral","lateral","parietal"): "streams",
    }
    roi_class = next((v for k, v in roi_class_map.items() if roi in k), None)
    if roi_class is None:
        raise ValueError(f"Unknown ROI: {roi}")
    h = hemisphere[0]
    algonauts = np.load(
        os.path.join(subj_dir, "roi_masks", f"{h}h.{roi_class}_challenge_space.npy"))
    roi_map = np.load(
        os.path.join(subj_dir, "roi_masks", f"mapping_{roi_class}.npy"),
        allow_pickle=True).item()
    roi_idx = list(roi_map.keys())[list(roi_map.values()).index(roi)]
    return algonauts == roi_idx


def load_roi_fmri(roi_name, hemisphere, subj, n_images=1000):
    """Load fMRI data for a specific subject and ROI.
    Returns array of shape (n_images, n_roi_voxels)."""
    subj_dir = os.path.join(FMRI_DIR, f"subj{subj:02d}")
    h = hemisphere[0]
    fmri     = np.load(os.path.join(subj_dir, f"{h}h_training_fmri.npy"))  # (n_imgs, n_vox)
    roi_mask = get_roi_mask(roi_name, hemisphere, subj_dir)
    return fmri[:n_images, roi_mask]   # (n_images, n_roi_voxels)

In [ ]:
# ── Settings: choose brain ROI and subject ────────────────────────────────────
# 'ventral' = ventral visual stream, a good proxy for IT cortex
# Also try: 'early' (primary visual), 'FFA-1' (faces), 'PPA' (places)
ROI_NAME   = "ventral"
HEMISPHERE = "left"
SUBJECT    = 1

try:
    roi_fmri = load_roi_fmri(ROI_NAME, HEMISPHERE, SUBJECT)
    print(f"fMRI data loaded: {roi_fmri.shape}")   # expected: (1000, n_voxels)
    HAVE_FMRI = True
except FileNotFoundError as e:
    print(f"[!] fMRI data not available yet: {e}")
    print("    Skipping encoding model. Run again once you have the data.")
    HAVE_FMRI = False

In [ ]:
if HAVE_FMRI:
    # ── 80/20 train/test split ────────────────────────────────────────────────
    # We train on 80% of images and predict on the held-out 20%.
    # This prevents overfitting — the test accuracy is honest.
    rng = np.random.default_rng(seed=42)
    idx = rng.permutation(N_IMAGES)
    n_train = int(0.8 * N_IMAGES)
    idx_train, idx_test = idx[:n_train], idx[n_train:]

    X_train = pixels_pca[idx_train]   # PCA pixel features, training images
    X_test  = pixels_pca[idx_test]    # PCA pixel features, test images
    Y_train = roi_fmri[idx_train]     # brain responses, training images
    Y_test  = roi_fmri[idx_test]      # brain responses, test images (ground truth)

    print(f"Training: {X_train.shape} -> {Y_train.shape}")
    print(f"Test:     {X_test.shape}  -> {Y_test.shape}")

    # ── Standardise fMRI before regression ───────────────────────────────────
    fmri_scaler = StandardScaler()
    Y_train_s = fmri_scaler.fit_transform(Y_train)
    Y_test_s  = fmri_scaler.transform(Y_test)

    # ── Fit Ridge regression (alpha chosen by 5-fold cross-validation) ────────
    # alpha controls the regularisation strength — larger alpha = simpler model
    alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    ridge  = RidgeCV(alphas=alphas, cv=5)
    ridge.fit(X_train, Y_train_s)
    print(f"Best alpha (regularisation): {ridge.alpha_:.4f}")

    # ── Evaluate on held-out test images ──────────────────────────────────────
    # R^2 score: 1 = perfect prediction, 0 = no better than predicting the mean
    r2 = ridge.score(X_test, Y_test_s)
    print(f"R2 on held-out test images: {r2:.3f}")
    print("(Positive R2 = model explains some variance in real brain responses)")

    # ── Predict responses for ALL 1000 images ─────────────────────────────────
    Y_pred_all = ridge.predict(pixels_pca)   # (1000, n_voxels)

    # ── Compute RDM from *predicted* brain responses ───────────────────────────
    linreg_rdm_vec = pdist(Y_pred_all, metric="correlation")
    linreg_rdm     = squareform(linreg_rdm_vec)

    np.save(os.path.join(RESULTS_DIR, "rdm_linreg.npy"),     linreg_rdm)
    np.save(os.path.join(RESULTS_DIR, "rdm_linreg_vec.npy"), linreg_rdm_vec)
    print("Saved linear regression RDM.")

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.matshow(linreg_rdm, cmap="viridis")
    plt.colorbar(im, ax=ax)
    ax.set_title(f"Linear regression RDM\n(pixel PCA -> {ROI_NAME} fMRI, subj {SUBJECT})")
    ax.set_xlabel("Image index")
    ax.set_ylabel("Image index")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "rdm_linreg.png"), dpi=100)
    plt.show()

## 5. Compare baselines to each other (sanity-check RSA)

Before comparing to brain data, check how similar the baselines are to each other.
We use **Spearman rank correlation** — the standard for RSA — on the upper triangle of each RDM.

> Pixel and PCA RDMs are expected to correlate highly (PCA is derived from pixels).
> Linear regression, if available, may differ more because it projects through brain responses.

In [ ]:
rdms = {"Pixel": pixel_rdm_vec, "PCA": pca_rdm_vec}
if HAVE_FMRI:
    rdms["LinReg"] = linreg_rdm_vec

model_names = list(rdms.keys())
n_m = len(model_names)

# Build model x model RSA matrix
rsa_matrix = np.zeros((n_m, n_m))
for i, ni in enumerate(model_names):
    for j, nj in enumerate(model_names):
        rsa_matrix[i, j], _ = spearmanr(rdms[ni], rdms[nj])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(rsa_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="Spearman r")
ax.set_xticks(range(n_m));  ax.set_xticklabels(model_names)
ax.set_yticks(range(n_m));  ax.set_yticklabels(model_names)
ax.set_title("RSA between baselines\n(Spearman r of RDM upper triangles)")
for i in range(n_m):
    for j in range(n_m):
        ax.text(j, i, f"{rsa_matrix[i,j]:.2f}", ha="center", va="center", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rsa_between_baselines.png"), dpi=100)
plt.show()

## 6. Summary

### Files saved to `results/`
| File | Description |
|---|---|
| `rdm_pixel.npy` | 1000×1000 RDM from raw pixel vectors |
| `rdm_pixel_vec.npy` | Upper triangle only (use for RSA comparisons) |
| `rdm_pca.npy` | 1000×1000 RDM from 50 PCA components |
| `rdm_pca_vec.npy` | Upper triangle only |
| `rdm_linreg.npy` | 1000×1000 RDM from Ridge regression predictions |
| `rdm_linreg_vec.npy` | Upper triangle only |

### What happens next (statistical analyses)
Once all group members finish their models, we will collect all `*_vec.npy` files and:
1. Load all model RDM upper-triangle vectors
2. Load the brain RDM vectors (fMRI and/or single-cell)
3. Compute **Spearman correlations** between each model RDM and the brain RDM
4. Run **permutation tests** (randomly shuffle image labels) to get p-values
5. Plot a bar chart: bar height = how well each model predicts the brain

The hypothesis: the beta-VAE bar should be **taller** than all baselines above.